# 🚲 千葉県の自転車盗データ分析
## 学生が自転車盗の被害に遭わないためには？

**メンバー：** 富田匠秋　坂井優希　馬場信吾

**調査したい問い：** 千葉県の自転車盗データでは、学生の被害はどのような場所・時間・施錠状況で多いのか。また、その結果からどのような防犯対策を提案できるか。

**データの出典：** 千葉県警察「自転車盗」オープンデータ（授業で配布されたCSV）

**データの説明：** 1行が自転車盗の記録1件を表す。地域、発生年月日・時刻、発生場所、被害者の年齢・職業、施錠状況など15項目が含まれている。公開分には2019～2025年の発生年月日と日付不明の記録が含まれるため、単純に「2025年だけの被害」とは扱わない。

## 1. CSVファイルを読み込む

CSVファイルをこのNotebookと同じフォルダに置き、ファイル名を **`bike.csv`** にする。千葉県警のCSVはCP932（Shift-JIS系）の文字コードなので、`encoding="cp932"` を指定する。

In [ ]:
# 読み込むファイルを設定する（Notebookと同じフォルダに保存）
filename = "bike.csv"
encoding = "cp932"

import csv
with open(filename, encoding=encoding, newline="") as file:
    incidents = list(csv.DictReader(file))

# 1行が1件の事件。件数を確認する
print("データ件数：", len(incidents))
print("列名：", list(incidents[0].keys()))

In [ ]:
# 後の集計で使う列名がCSVにあるか確認する（なければ原因が分かるメッセージを出す）
required_columns = ["市区町村（発生地）", "被害者の職業", "施錠関係",
                    "発生場所の詳細", "発生時（始期）"]
for column in required_columns:
    if column not in incidents[0]:
        raise KeyError("CSVに列「" + column + "」がありません。上の列名と見比べてください。")
print("必要な列がすべてあります")

読み込んだ件数が **10,054件** になっていることを確認する。もし違う場合は、CSVのファイル名・保存場所・文字コードを確認する。

In [ ]:
# 元データの最初の3件を確認する
for row in incidents[:3]:
    print(row)

## 2. 市区町村別の件数を集計する

`for`文で全レコードを1件ずつ確認し、市区町村ごとに件数を数える。ここでの件数は記録された事件数であり、人口や駐輪台数を考慮した被害率ではない。

In [ ]:
municipality_counts = {}

for row in incidents:
    area = row["市区町村（発生地）"]
    if area in municipality_counts:
        municipality_counts[area] += 1
    else:
        municipality_counts[area] = 1

# 件数の多い順に並べて上位10地域を表示
municipality_ranking = sorted(municipality_counts.items(),
                              key=lambda item: item[1],
                              reverse=True)

print("市区町村別 自転車盗件数 上位10")
for area, count in municipality_ranking[:10]:
    print(area, count, "件")

### 市区町村別ランキングのグラフ

上位10地域の件数を横棒グラフにする。

In [ ]:
import matplotlib.pyplot as plt

top_areas = municipality_ranking[:10]
area_names = [item[0] for item in top_areas]
area_values = [item[1] for item in top_areas]

plt.figure(figsize=(10, 6))
plt.barh(area_names[::-1], area_values[::-1])
plt.xlabel("Number of recorded bicycle thefts (cases)")
plt.ylabel("Municipality / ward")
plt.title("Top 10 areas by recorded bicycle thefts")
plt.tight_layout()
plt.show()

## 3. 学生の被害に絞って調べる

被害者の職業が「小学生・中学生・高校生・大学生」の記録を、`for`文で取り出す。その後、施錠状況・発生場所の詳細・発生時刻を集計する。

In [ ]:
student_jobs = ["小学生", "中学生", "高校生", "大学生"]
student_records = []

for row in incidents:
    if row["被害者の職業"] in student_jobs:
        student_records.append(row)

print("学生の被害記録：", len(student_records), "件")
print("全体に占める割合：{:.1f}%".format(
    len(student_records) / len(incidents) * 100
))

# 学生の職業別件数
job_counts = {}
for row in student_records:
    job = row["被害者の職業"]
    if job in job_counts:
        job_counts[job] += 1
    else:
        job_counts[job] = 1

print("\n学生の職業別件数")
for job, count in sorted(job_counts.items(), key=lambda item: item[1], reverse=True):
    print(job, count, "件")

### 学生の施錠状況を集計する

In [ ]:
student_lock_counts = {}
for row in student_records:
    lock = row["施錠関係"]
    if lock in student_lock_counts:
        student_lock_counts[lock] += 1
    else:
        student_lock_counts[lock] = 1

print("学生の施錠状況")
for lock, count in student_lock_counts.items():
    print(lock, count, "件（{:.1f}%）".format(
        count / len(student_records) * 100
    ))

# 学生の発生場所の詳細
student_place_counts = {}
for row in student_records:
    place = row["発生場所の詳細"]
    if place in student_place_counts:
        student_place_counts[place] += 1
    else:
        student_place_counts[place] = 1

print("\n学生の発生場所の詳細")
for place, count in student_place_counts.items():
    print(place, count, "件（{:.1f}%）".format(
        count / len(student_records) * 100
    ))

## 4. 学生の被害が発生した時刻を調べる

時刻が「不明」の記録は時間別グラフから除外する。ここでは時刻の文字列を整数に変換せず、`"07"` などの値をそのまま集計する。

In [ ]:
student_hour_counts = {}
unknown_hour_count = 0

for row in student_records:
    hour = row["発生時（始期）"]
    if hour == "不明" or hour == "":
        unknown_hour_count += 1
    else:
        if hour in student_hour_counts:
            student_hour_counts[hour] += 1
        else:
            student_hour_counts[hour] = 1

# 00時から23時の順に表示
print("学生の時間別件数")
for hour in sorted(student_hour_counts.keys(), key=int):
    print(hour + "時", student_hour_counts[hour], "件")

print("時刻不明：", unknown_hour_count, "件")

In [ ]:
hours = sorted(student_hour_counts.keys(), key=int)
hour_values = [student_hour_counts[hour] for hour in hours]

plt.figure(figsize=(11, 5))
plt.bar(hours, hour_values)
plt.xlabel("Hour")
plt.ylabel("Number of cases")
plt.title("Bicycle thefts involving students by hour")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 下の「分析結果」の文章に書いた数値を、実行結果から確認する
total = len(incidents)
students = len(student_records)
print("全体:", total, "件 / 学生:", students, "件（{:.1f}%）".format(students / total * 100))

no_lock = student_lock_counts.get("施錠せず", 0)
print("学生で施錠せず:", no_lock, "件（{:.1f}%）".format(no_lock / students * 100))

parking = student_place_counts.get("駐車（輪）場", 0)
print("学生で駐車（輪）場:", parking, "件（{:.1f}%）".format(parking / students * 100))

# 時間別で最も多い時刻（for文で探す）
peak_hour, peak_count = None, 0
for hour, count in student_hour_counts.items():
    if count > peak_count:
        peak_hour, peak_count = hour, count
print("学生の時間別で最多:", peak_hour, "時", peak_count, "件")

## 5. 分析結果から分かったこと

以下はこのCSVから確認できる数値をもとにした発見である。Notebookを実行した結果と数字が一致するか確認する。

1. **学生の被害が多い。** 小学生・中学生・高校生・大学生の記録は合計5,044件で、全10,054件の約50.2%を占める。
2. **学生の記録では施錠されていない自転車が多い。** 施錠せずは3,769件で、学生の記録の約74.7%である。ただし、このデータだけで施錠による被害防止効果を因果関係として証明することはできない。
3. **学生の被害は駐車（輪）場の記録が多い。** 発生場所の詳細が「駐車（輪）場」の記録は4,247件で、学生の記録の約84.2%である。
4. **学生の時間別では7時が最多。** 7時は540件で、17時は471件、18時は440件である。登校・下校時間帯に近い時間に多いことが読み取れるが、個々の事件の状況までは分からない。

### 防犯の提案

学校や駅の駐輪場では、短時間でも必ず鍵をかけることを呼びかけ、備え付けの鍵に加えてワイヤーロックなどを使う「ツーロック」を勧める。また、登校前の朝や放課後の時間帯に、駐輪場で施錠を確認するポスターや声かけを行う。学生の被害記録の約84.2%が駐車（輪）場の詳細区分だったため、駐輪場での注意喚起を重点的に行うことがデータに基づく提案となる。

## 6. 集計結果の確認とデータの限界

### 件数の照合
学生の職業別件数の合計が、学生として抽出した件数と一致するか確認する。また、施錠状況の合計も学生の件数と一致するか確認する。

In [ ]:
print("学生の抽出件数：", len(student_records))
print("職業別件数の合計：", sum(job_counts.values()))
print("施錠状況の合計：", sum(student_lock_counts.values()))
print("場所詳細の合計：", sum(student_place_counts.values()))

# 全データの施錠状況も照合する
all_lock_counts = {}
for row in incidents:
    lock = row["施錠関係"]
    if lock in all_lock_counts:
        all_lock_counts[lock] += 1
    else:
        all_lock_counts[lock] = 1

print("\n全データの施錠状況：", all_lock_counts)
print("全データの施錠状況の合計：", sum(all_lock_counts.values()))